Завдання 1. Читання, вибір колонок і фільтрація

In [0]:
trips=spark.table("samples.nyctaxi.trips")

In [0]:
trips_clean=(
    trips
    .select(
        "tpep_pickup_datetime", "trip_distance", "fare_amount", "pickup_zip", "dropoff_zip"
    )
    .filter(
        trips.pickup_zip.isNotNull()
        & trips.dropoff_zip.isNotNull()  
    )
    .filter(
        (trips.trip_distance>0)
        & (trips.fare_amount>0)
    )
)

trips_clean.show(5)

+--------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|          2.6|       18.5|     10199|      10022|
+--------------------+-------------+-----------+----------+-----------+
only showing top 5 rows


Завдання 2. Нові колонки

In [0]:
from pyspark.sql import functions as F
trips_prepared=(
    trips_clean
    .select(
        "tpep_pickup_datetime", "trip_distance", "fare_amount", "pickup_zip", "dropoff_zip"
    )
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("fare_per_mile", F.col("fare_amount")/F.col("trip_distance"))
)

trips_prepared.show(5)

+--------------------+-------------+-----------+----------+-----------+------------+------------------+
|tpep_pickup_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|pickup_month|     fare_per_mile|
+--------------------+-------------+-----------+----------+-----------+------------+------------------+
| 2016-02-13 21:47:53|          1.4|        8.0|     10103|      10110|           2| 5.714285714285714|
| 2016-02-13 18:29:09|         1.31|        7.5|     10023|      10023|           2|5.7251908396946565|
| 2016-02-06 19:40:58|          1.8|        9.5|     10001|      10018|           2| 5.277777777777778|
| 2016-02-12 19:06:43|          2.3|       11.5|     10044|      10111|           2|               5.0|
| 2016-02-23 10:27:56|          2.6|       18.5|     10199|      10022|           2| 7.115384615384615|
+--------------------+-------------+-----------+----------+-----------+------------+------------------+
only showing top 5 rows


Завдання 3. Запис у файли тома і читання файлів

In [0]:
path="/Volumes/workspace/default/taxi_files/trips_prepared"

spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.taxi_files
""")

(
    trips_prepared
    .write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(path)
)

trips_from_files = spark.read.parquet(path)

display(dbutils.fs.ls(path))

trips_from_files.show(5)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/_SUCCESS,_SUCCESS,0,1795022497000
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/pickup_month=1/,pickup_month=1/,0,1795022497463
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/pickup_month=2/,pickup_month=2/,0,1795022497463


+--------------------+-------------+-----------+----------+-----------+------------------+------------+
|tpep_pickup_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|     fare_per_mile|pickup_month|
+--------------------+-------------+-----------+----------+-----------+------------------+------------+
| 2016-02-13 21:47:53|          1.4|        8.0|     10103|      10110| 5.714285714285714|           2|
| 2016-02-13 18:29:09|         1.31|        7.5|     10023|      10023|5.7251908396946565|           2|
| 2016-02-06 19:40:58|          1.8|        9.5|     10001|      10018| 5.277777777777778|           2|
| 2016-02-12 19:06:43|          2.3|       11.5|     10044|      10111|               5.0|           2|
| 2016-02-23 10:27:56|          2.6|       18.5|     10199|      10022| 7.115384615384615|           2|
+--------------------+-------------+-----------+----------+-----------+------------------+------------+
only showing top 5 rows


Завдання 4

In [0]:
pickups_by_zip=(
    trips_from_files
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trips_started"),
        F.sum("fare_amount").alias("total_fare"),
        F.avg("fare_per_mile").alias("avg_fare_per_mile")
    )
    .orderBy("pickup_zip")
)

pickups_by_zip.show()

+----------+-------------+----------+------------------+
|pickup_zip|trips_started|total_fare| avg_fare_per_mile|
+----------+-------------+----------+------------------+
|      7002|            2|      21.0| 7.492672203224231|
|      7030|            1|      40.0|  6.34920634920635|
|      7087|            1|      31.0|3.5755478662053055|
|      7311|            1|      60.0|              30.0|
|      7718|            1|       5.0|              6.25|
|      7737|            1|      15.5| 5.308219178082192|
|     10001|         1227|  13028.01| 6.272036894787021|
|     10002|          611|    7439.5|5.4275164904637325|
|     10003|         1180|   12963.0| 7.608240110891416|
|     10005|          101|    1602.5| 4.740302442734421|
|     10006|          124|    1899.5| 4.925506643028728|
|     10007|          167|    2319.0| 5.422377976971116|
|     10009|          658|    7320.5| 6.113875566642556|
|     10011|         1128|   12319.0| 6.107694197599891|
|     10012|          831|    9

In [0]:
dropoffs_by_zip=(
    trips_from_files
    .groupBy("dropoff_zip")
    .agg(
        F.count("*").alias("trips_finished")
    )
    .orderBy("dropoff_zip")
)

dropoffs_by_zip.show()

+-----------+--------------+
|dropoff_zip|trips_finished|
+-----------+--------------+
|       7002|             2|
|       7008|             1|
|       7024|             2|
|       7030|             4|
|       7087|             1|
|       7114|            28|
|       7201|             1|
|       7302|             1|
|       7304|             1|
|       7310|             1|
|       7311|             3|
|       7423|             1|
|       7737|             1|
|       7758|             1|
|       7901|             1|
|      10001|          1069|
|      10002|           486|
|      10003|           929|
|      10005|           102|
|      10006|           132|
+-----------+--------------+
only showing top 20 rows


Завдання 5. Зʼєднання і значення null

Завдання 5.1. Ліве зʼєднання

In [0]:
zip_joined=(
    dropoffs_by_zip
    .join(pickups_by_zip, dropoffs_by_zip.dropoff_zip==pickups_by_zip.pickup_zip, "left")
)

zip_joined.show()

+-----------+--------------+----------+-------------+----------+------------------+
|dropoff_zip|trips_finished|pickup_zip|trips_started|total_fare| avg_fare_per_mile|
+-----------+--------------+----------+-------------+----------+------------------+
|      10022|           482|     10022|          517|    5090.0| 6.205664089593452|
|      10002|           486|     10002|          611|    7439.5|5.4275164904637325|
|      10103|           472|     10103|          580|   7285.51| 6.263780619520343|
|      10278|           160|     10278|          167|    2169.5| 5.686703578072423|
|      10280|            91|     10280|          103|    1681.5| 5.206186256436832|
|      10465|             3|      NULL|         NULL|      NULL|              NULL|
|       7302|             1|      NULL|         NULL|      NULL|              NULL|
|      10710|             1|      NULL|         NULL|      NULL|              NULL|
|      11575|             1|      NULL|         NULL|      NULL|            

Завдання 5.2. Індекси без посадок

In [0]:
zips_without_pickups=(
    zip_joined
    .filter(
        zip_joined.pickup_zip.isNull()
    )
)

zips_without_pickups.show()

+-----------+--------------+----------+-------------+----------+-----------------+
|dropoff_zip|trips_finished|pickup_zip|trips_started|total_fare|avg_fare_per_mile|
+-----------+--------------+----------+-------------+----------+-----------------+
|      10465|             3|      NULL|         NULL|      NULL|             NULL|
|       7302|             1|      NULL|         NULL|      NULL|             NULL|
|      10710|             1|      NULL|         NULL|      NULL|             NULL|
|      11575|             1|      NULL|         NULL|      NULL|             NULL|
|       7304|             1|      NULL|         NULL|      NULL|             NULL|
|      10462|             3|      NULL|         NULL|      NULL|             NULL|
|      11415|             2|      NULL|         NULL|      NULL|             NULL|
|      10463|             9|      NULL|         NULL|      NULL|             NULL|
|      11360|             2|      NULL|         NULL|      NULL|             NULL|
|   

Завдання 5.3. Підсумковий звіт

In [0]:
zip_report = (
    zip_joined
    .fillna(0, subset=['trips_started', 'total_fare'])
    .withColumn(
        'net_flow', F.col('trips_finished') - F.col('trips_started')
    )
    .withColumn('zip', F.col('dropoff_zip'))
    .select(
        'zip',
        'trips_finished',
        'trips_started',
        'net_flow',
        'total_fare',
        'avg_fare_per_mile'
    )
    .orderBy('zip')
)

zip_report.show()

+-----+--------------+-------------+--------+----------+------------------+
|  zip|trips_finished|trips_started|net_flow|total_fare| avg_fare_per_mile|
+-----+--------------+-------------+--------+----------+------------------+
| 7002|             2|            2|       0|      21.0| 7.492672203224231|
| 7008|             1|            0|       1|       0.0|              NULL|
| 7024|             2|            0|       2|       0.0|              NULL|
| 7030|             4|            1|       3|      40.0|  6.34920634920635|
| 7087|             1|            1|       0|      31.0|3.5755478662053055|
| 7114|            28|            0|      28|       0.0|              NULL|
| 7201|             1|            0|       1|       0.0|              NULL|
| 7302|             1|            0|       1|       0.0|              NULL|
| 7304|             1|            0|       1|       0.0|              NULL|
| 7310|             1|            0|       1|       0.0|              NULL|
| 7311|     

Завдання 6. Конвеєр одним ланцюжком і запит SQL

In [0]:
pickups_chain=(
    spark.table("samples.nyctaxi.trips")
    .select(
        "tpep_pickup_datetime",
        "trip_distance",
        "fare_amount",
        "pickup_zip",
        "dropoff_zip"
    )
    .filter(
        F.col("pickup_zip").isNotNull()
        & F.col("dropoff_zip").isNotNull()  
    )
    .filter(
        (F.col("trip_distance")>0)
        & (F.col("fare_amount")>0) 
    )
    .withColumn("fare_per_mile", F.col("fare_amount")/F.col("trip_distance"))
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trips_started"),
        F.sum("fare_amount").alias("total_fare"),
        F.avg("fare_per_mile").alias("avg_fare_per_mile")
    )
    .orderBy("pickup_zip")
)

display(pickups_chain)

pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.492672203224231
7030,1,40.0,6.34920634920635
7087,1,31.0,3.5755478662053055
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.308219178082192
10001,1227,13028.01,6.27203689478702
10002,611,7439.5,5.427516490463737
10003,1180,12963.0,7.608240110891412
10005,101,1602.5,4.74030244273442


In [0]:
pickups_sql = spark.sql("""
    SELECT 
        pickup_zip, 
        COUNT(*) AS trips_started, 
        SUM(fare_amount) AS total_fare, 
        AVG(fare_amount/trip_distance) AS avg_fare_per_mile
    FROM samples.nyctaxi.trips
    WHERE pickup_zip IS NOT NULL AND dropoff_zip IS NOT NULL AND trip_distance > 0 AND fare_amount > 0
    GROUP BY pickup_zip
    ORDER BY pickup_zip                     
""")

display(pickups_sql)

pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.492672203224231
7030,1,40.0,6.34920634920635
7087,1,31.0,3.5755478662053055
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.308219178082192
10001,1227,13028.01,6.27203689478702
10002,611,7439.5,5.427516490463737
10003,1180,12963.0,7.608240110891412
10005,101,1602.5,4.74030244273442


Завдання 7. Запис у таблицю і прибирання

Завдання 7.1. Запис у таблицю

In [0]:
(
    zip_report.write
    .mode("overwrite")
    .saveAsTable("workspace.default.zip_report")
)

zip_report_from_table = (
    spark.table("workspace.default.zip_report")
    .orderBy("zip")
)

display(zip_report_from_table)

zip,trips_finished,trips_started,net_flow,total_fare,avg_fare_per_mile
7002,2,2,0,21.0,7.492672203224231
7008,1,0,1,0.0,null
7024,2,0,2,0.0,null
7030,4,1,3,40.0,6.34920634920635
7087,1,1,0,31.0,3.5755478662053055
7114,28,0,28,0.0,null
7201,1,0,1,0.0,null
7302,1,0,1,0.0,null
7304,1,0,1,0.0,null
7310,1,0,1,0.0,null


Завдання 7.2. Прибирання

In [0]:
spark.sql("""
    DROP TABLE IF EXISTS workspace.default.zip_report
""")

dbutils.fs.rm(path, True)

spark.sql("""
    DROP VOLUME IF EXISTS workspace.default.taxi_files
""")

DataFrame[]